# NLPG Practical 2 — Grammars: generate, recognise, parse, and learn from a treebank

In the lecture we treated a grammar as a **finite description of possible structure**. Here you will use grammars in several directions:

1. **generate** sentences licensed by a grammar;
2. **recognise** whether a sentence is licensed;
3. **parse** a sentence to recover its structure(s);
4. change rules and observe **overgeneration, undergeneration and ambiguity**;
5. use recursion to describe open-ended structure;
6. extract a CFG from the **Penn Treebank** and test what it learned from a finite sample.

You should already have worked through the short Penn Treebank annotation notebook.

There is nothing to submit, and you do not need to edit this notebook to record answers. As you work, take your own notes on the required activities and what you observe. Keep your notes and bring them to this week's summing-up lecture (L2), where we may compare and discuss your findings.

> **Main question: Where is the generalisation?**

## Setup

In [ ]:
import random
from collections import Counter, defaultdict
import nltk
from nltk import CFG, Nonterminal
from nltk.parse import ChartParser
from nltk.parse.generate import generate

import os
from dotenv import load_dotenv

load_dotenv()
nltk.download("treebank", download_dir=os.environ["NLTK_DATA"])
from nltk.corpus import treebank

# Part A — One grammar, three operations

We begin with a deliberately small grammar. Because every rule is visible, we can reason about exactly what it licenses.

In [ ]:
grammar = CFG.fromstring(r'''
S  -> NP VP
NP -> Det N
VP -> V NP
Det -> 'the' | 'a'
N   -> 'dog' | 'cat' | 'student'
V   -> 'chased' | 'saw'
''')
parser = ChartParser(grammar)
print(grammar)

## A1. Generate

Generation starts with the grammar and asks: **What sentences can this grammar produce?**

NLTK's generator needs a depth bound because grammars may contain recursion.

In [ ]:
for sent in list(generate(grammar, depth=6))[:20]:
    print(" ".join(sent))

### Required activity 1A

Inspect the output. Which parts vary? Which combinations are impossible? **Where in the grammar are those restrictions expressed?**

## A2. Recognise

Recognition asks: **Does the grammar license this sentence?**

Here a sentence is recognised if the parser finds at least one parse.

In [ ]:
def recognise(parser, sentence):
    tokens = sentence.split()
    try:
        return next(parser.parse(tokens), None) is not None
    except ValueError:
        return False

tests = [
    "the dog chased the cat",
    "a student saw the dog",
    "dog the chased cat the",
    "the cat admired the student",
]
for sentence in tests:
    print(f"{recognise(parser, sentence)!s:5}  {sentence}")

A rejection can have different causes. *admired*, for example, is absent from this grammar's lexicon.

### Required activity 1B

Add one sentence you predict will be accepted and one you predict will be rejected. **Predict first; run second.** If you are wrong, identify the rule—or missing rule—that explains why.

In [ ]:
my_tests = [
    # "your sentence here",
    # "another sentence here",
]
for sentence in my_tests:
    print(f"{recognise(parser, sentence)!s:5}  {sentence}")

## A3. Parse

Parsing asks: **If the sentence is licensed, what structure—or structures—does the grammar assign?**

In [ ]:
sentence = "the dog chased a cat"
parses = list(parser.parse(sentence.split()))
print("Number of parses:", len(parses))
for t in parses:
    t.pretty_print()

The same grammar supports three operations:

| Operation | Question |
|---|---|
| Generate | What strings can the grammar produce? |
| Recognise | Is this string licensed? |
| Parse | What structure(s) does the grammar assign? |

# Part B — Changing the grammar changes what is possible

Consider:

> **I saw the man with the telescope**

It has two familiar readings: I used the telescope, or the man had the telescope. A small grammar can license both.

In [ ]:
ambiguous_grammar = CFG.fromstring(r'''
S  -> NP VP
NP -> Pronoun
NP -> Det N
NP -> NP PP
VP -> V NP
VP -> VP PP
PP -> P NP
Pronoun -> 'I'
Det -> 'the'
N   -> 'man' | 'telescope'
V   -> 'saw'
P   -> 'with'
''')
ambiguous_parser = ChartParser(ambiguous_grammar)
sentence = "I saw the man with the telescope"
parses = list(ambiguous_parser.parse(sentence.split()))
print("Number of parses:", len(parses))
for i, t in enumerate(parses, 1):
    print(f"\nPARSE {i}")
    t.pretty_print()

### Required activity 2 — Remove an ambiguity

The two attachment rules are:

```text
NP -> NP PP
VP -> VP PP
```

Make two modified grammars: one without the NP attachment rule and one without the VP attachment rule.

Before running each, predict:

- how many parses remain;
- which interpretation remains;
- what useful constructions deleting the rule might accidentally exclude.

That last question is **undergeneration**.

In [ ]:
# Copy the grammar above twice and remove ONE attachment rule in each.
# Then construct ChartParser objects and parse the same sentence.

> **Ambiguity is not necessarily a parser error. The grammar itself may license more than one structure.**

# Part C — Recursion creates open-ended structure

The rule `NP -> NP PP` is recursive: an NP can form part of a larger NP.

In [ ]:
recursive_grammar = CFG.fromstring(r'''
S  -> NP VP
NP -> Det N
NP -> NP PP
VP -> V
PP -> P NP
Det -> 'the'
N   -> 'dog' | 'cat' | 'student' | 'telescope'
V   -> 'slept'
P   -> 'near' | 'beside' | 'with'
''')
recursive_parser = ChartParser(recursive_grammar)

examples = [
    "the dog slept",
    "the dog near the cat slept",
    "the dog near the cat beside the student slept",
    "the dog near the cat beside the student with the telescope slept",
]
for sentence in examples:
    parses = list(recursive_parser.parse(sentence.split()))
    print(f"{len(parses):3} parse(s)  {sentence}")

### Required activity 3 — Follow the recursion

Display one parse of the longest example and locate each application of `NP -> NP PP`.

In [ ]:
sentence = examples[-1]
parses = list(recursive_parser.parse(sentence.split()))
if parses:
    parses[0].pretty_print()

Consider:

- The grammar is finite. Why is there no fixed maximum number of PPs imposed by it?
- Why does a program nevertheless need a depth bound for generation?
- Does “recursive” mean every generated sentence will be sensible English?

The intended claim is **unbounded in principle**, not that humans produce infinitely long sentences.

# Part D — From an annotated corpus to a grammar

So far we wrote the grammar ourselves. Now reverse the direction:

> **Can we obtain grammar rules from structures observed in a corpus?**

The Penn Treebank contains constituency trees, and NLTK can extract their productions.

In [ ]:
t = treebank.parsed_sents()[0]
t.pretty_print()

print("\nFIRST 20 PRODUCTIONS")
for p in t.productions()[:20]:
    print(p)

Productions include both **structural rules** such as `NP -> DT NN` and **lexical productions** such as `NN -> 'market'`.

NLTK represents both as productions, but it is useful to distinguish structural grammar from lexicon.

In [ ]:
def is_lexical(production):
    return all(isinstance(x, str) for x in production.rhs())

structural = [p for p in t.productions() if not is_lexical(p)]
lexical = [p for p in t.productions() if is_lexical(p)]

print("STRUCTURAL")
for p in structural[:12]: print(" ", p)
print("\nLEXICAL")
for p in lexical[:12]: print(" ", p)

## D1. Extract a grammar from a training sample

We use Treebank trees whose root category is `S`, reserving later trees as held-out evidence.

In [ ]:
s_trees = [t for t in treebank.parsed_sents() if t.label() == "S"]
TRAIN_SIZE, TEST_SIZE = 800, 100
train_trees = s_trees[:TRAIN_SIZE]
test_trees = s_trees[TRAIN_SIZE:TRAIN_SIZE + TEST_SIZE]
print("Training trees:", len(train_trees))
print("Held-out trees:", len(test_trees))

In [ ]:
training_productions = []
for t in train_trees:
    training_productions.extend(t.productions())

production_counts = Counter(training_productions)
unique_productions = list(production_counts)
extracted_grammar = CFG(Nonterminal("S"), unique_productions)

structural_rules = [p for p in unique_productions if not is_lexical(p)]
lexical_rules = [p for p in unique_productions if is_lexical(p)]

print("Observed production tokens:", len(training_productions))
print("Unique productions:", len(unique_productions))
print("Unique structural productions:", len(structural_rules))
print("Unique lexical productions:", len(lexical_rules))

### Required activity 4A — What has extraction done?

Inspect the most frequent productions.

In [ ]:
for production, count in production_counts.most_common(25):
    kind = "LEX" if is_lexical(production) else "SYN"
    print(f"{count:5}  {kind}  {production}")

Consider:

- Why are there fewer **unique productions** than production occurrences?
- What information did we throw away when we converted the counts into an ordinary unweighted CFG?
- Which parts came from annotation, and which came directly from observed words?

Do **not** calculate probabilities yet. Simply notice that the counts exist.

## D2. Generate from the extracted grammar

Every rule was observed in an annotated sentence. But an ordinary CFG can **recombine** those rules in combinations that were never observed.

The helper below makes a random derivation with a maximum depth. You do not need to understand its implementation.

In [ ]:
by_lhs = defaultdict(list)
for p in extracted_grammar.productions():
    by_lhs[p.lhs()].append(p)

INF = 10**9
min_depth = {nt: INF for nt in by_lhs}
changed = True
while changed:
    changed = False
    for lhs, rules in by_lhs.items():
        for p in rules:
            ds = [min_depth[x] for x in p.rhs() if isinstance(x, Nonterminal)]
            if any(d == INF for d in ds):
                continue
            needed = 1 + (max(ds) if ds else 0)
            if needed < min_depth[lhs]:
                min_depth[lhs] = needed
                changed = True

def random_sentence(grammar, max_depth=12, seed=None):
    rng = random.Random(seed)
    def expand(symbol, depth_left):
        if isinstance(symbol, str):
            return [symbol]
        candidates = []
        for p in by_lhs[symbol]:
            nts = [x for x in p.rhs() if isinstance(x, Nonterminal)]
            required = 1 + max((min_depth[x] for x in nts), default=0)
            if required <= depth_left:
                candidates.append(p)
        if not candidates:
            raise RuntimeError(f"No terminating expansion for {symbol}")
        p = rng.choice(candidates)
        words = []
        for x in p.rhs():
            words.extend(expand(x, depth_left - 1))
        return words
    return expand(grammar.start(), max_depth)

In [ ]:
for seed in range(5):
    try:
        print(" ".join(random_sentence(extracted_grammar, max_depth=12, seed=seed)))
    except RuntimeError as e:
        print("Generation failed:", e)

### Required activity 4B — Real ingredients, peculiar combinations

Every production above came from the training treebank, yet generated sentences may be peculiar. Explain why.

Consider what we preserved:

- observed syntactic productions;
- observed lexical productions;

and what we discarded:

- how often alternatives occurred;
- many lexical preferences and dependencies;
- wider semantic and discourse constraints.

## D3. Test the extracted grammar on held-out trees

A grammar extracted from a finite sample need not cover another sample from the same treebank.

Two obvious reasons are:

1. a held-out sentence contains a **word** unseen in training;
2. its gold tree contains a **production** unseen in training.

We can diagnose these directly from the held-out annotation without making a large parser search.

In [ ]:
known_words = {
    x for p in lexical_rules for x in p.rhs()
    if isinstance(x, str)
}
known_productions = set(unique_productions)

def diagnose_tree(t):
    unseen_words = sorted(set(t.leaves()) - known_words)
    unseen_productions = [p for p in t.productions() if p not in known_productions]
    return unseen_words, unseen_productions

short_test_trees = [t for t in test_trees if len(t.leaves()) <= 15][:10]

for i, t in enumerate(short_test_trees, 1):
    unseen_words, unseen_productions = diagnose_tree(t)
    print(f"\n{i}. {' '.join(t.leaves())}")
    print("   unseen word types:", len(unseen_words))
    print("   unseen productions:", len(unseen_productions))
    if unseen_words:
        print("   example unseen words:", unseen_words[:5])
    if unseen_productions:
        print("   example unseen production:", unseen_productions[0])

### Required activity 4C — What does a grammar learned from a sample know?

Choose two held-out sentences:

- one with an unseen word;
- one with an unseen structural production, if available.

Identify precisely what the training-derived grammar is missing.

Then ask: if we simply enlarge the training sample, which problems are likely to shrink—and which modelling limitations remain?

This is the Week 1 idea in a new representation:

**finite observable evidence → model → behaviour on new evidence**

# Bringing the practical together

You have used a CFG in four related ways:

1. **Generation:** grammar → sentences
2. **Recognition:** grammar + sentence → licensed / not licensed
3. **Parsing:** grammar + sentence → structure(s)
4. **Extraction:** annotated structures → grammar

The Treebank tells us not only **which productions occurred**, but **how often they occurred**. Our extracted CFG retained the alternatives but discarded those frequencies.

> **If different grammatical choices occur with very different frequencies, should a model treat them all alike?**

We leave that question open here.

## What to bring to L2

Keep brief notes on:

- one example of overgeneration or undergeneration;
- what happened to PP attachment when you changed the grammar;
- what recursion allowed;
- one reason the Treebank-derived grammar failed on held-out evidence;
- why observed productions can still recombine into peculiar sentences.

We may compare and discuss your findings in the summing-up lecture.